In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsssmartdata2698";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas gold del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.orders_raw (
    order_id STRING,
    customer_id STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/orders_raw"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.customers_raw (
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix INT,
    customer_city STRING,
    customer_state STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/customers_raw"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.order_items_raw (
    order_id STRING,
    order_item_id INT,
    product_id STRING,
    seller_id STRING,
    shipping_limit_date TIMESTAMP,
    price DOUBLE,
    freight_value DOUBLE,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/order_items_raw'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.products_raw (
    product_id STRING,
    product_category_name STRING,
    product_name_lenght INT,
    product_description_lenght INT,
    product_photos_qty INT,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/products_raw'

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.payments_raw (
    order_id STRING,
    payment_sequential INT,
    payment_type STRING,
    payment_installments INT,
    payment_value DOUBLE,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/payments_raw';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.category_translation_raw (
    product_category_name STRING,
    product_category_name_english STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storageName}.dfs.core.windows.net/category_translation_raw'

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.olist_orders_enriched (
    order_id STRING,
    customer_id STRING,
    customer_unique_id STRING,
    customer_city STRING,
    customer_state STRING,
    customer_region STRING,
    order_status STRING,
    order_purchase_timestamp TIMESTAMP,
    order_purchase_date DATE,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    delivery_days INT,
    is_late_delivery BOOLEAN,
    delivery_status STRING,
    order_item_id INT,
    product_id STRING,
    seller_id STRING,
    shipping_limit_date TIMESTAMP,
    price DOUBLE,
    freight_value DOUBLE,
    product_category_name STRING,
    product_category_name_english STRING,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE,
    product_volume_cm3 DOUBLE,
    payment_type STRING,
    payment_installments INT,
    payment_value DOUBLE,
    transformation_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@${storageName}.dfs.core.windows.net/olist_orders_enriched';

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.olist_sales_summary (
    year INT,
    month INT,
    year_month STRING,
    customer_state STRING,
    customer_region STRING,
    product_category_name_english STRING,
    payment_type STRING,
    delivery_status STRING,
    total_orders BIGINT,
    total_customers BIGINT,
    total_items_sold BIGINT,
    total_revenue DOUBLE,
    total_freight DOUBLE,
    total_payment_value DOUBLE,
    average_item_price DOUBLE,
    average_payment_value DOUBLE,
    average_delivery_days DOUBLE,
    late_orders BIGINT,
    late_delivery_rate DOUBLE
)
USING DELTA
LOCATION 'abfss://golden@${storageName}.dfs.core.windows.net/olist_sales_summary';